# **Create Dimensions and Facts**

In [1]:
#import project lip
from pyspark.sql.types import *
from delta.tables import DeltaTable

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 3, Finished, Available, Finished, False)

In [2]:
last_load_date = spark.sql("""
    SELECT Last_Load_Date 
    FROM GamingPlatform_STG.dbo.ETL_Metadata
    WHERE Stage = 'STG_Gaming'
""").collect()[0]["Last_Load_Date"]

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 4, Finished, Available, Finished, False)

In [3]:
# last_load_date='1999-01-01 00:00:00'

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 5, Finished, Available, Finished, False)

## Load Data from all GamingPlatform_STG tables into DataFrames

In [4]:
# Load Data from all GamingPlatform_STG tables into DataFrames
df_cities = spark.sql(f"""
    SELECT * FROM GamingPlatform_STG.dbo.cities
    WHERE ModifiedDate > '{last_load_date}'
""")

df_countries = spark.sql(f"""
    SELECT * FROM GamingPlatform_STG.dbo.countries
    WHERE ModifiedDate > '{last_load_date}'
""")

df_game_genres = spark.sql(f"""
    SELECT * FROM GamingPlatform_STG.dbo.game_genres
    WHERE ModifiedDate > '{last_load_date}'
""")

df_game_metadata = spark.sql(f"""
    SELECT * FROM GamingPlatform_STG.dbo.game_metadata
    WHERE ModifiedDate > '{last_load_date}'
""")

df_game_prices = spark.sql(f"""
    SELECT * FROM GamingPlatform_STG.dbo.game_prices
    WHERE ModifiedDate > '{last_load_date}'
""")

df_game_sessions = spark.sql(f"""
    SELECT * FROM GamingPlatform_STG.dbo.game_sessions
    WHERE ModifiedDate > '{last_load_date}'
""")

df_game_titles = spark.sql(f"""
    SELECT * FROM GamingPlatform_STG.dbo.game_titles
    WHERE ModifiedDate > '{last_load_date}'
""")

df_orders = spark.sql(f"""
    SELECT * FROM GamingPlatform_STG.dbo.orders
    WHERE ModifiedDate > '{last_load_date}'
""")

df_states = spark.sql(f"""
    SELECT * FROM GamingPlatform_STG.dbo.states
    WHERE ModifiedDate > '{last_load_date}'
""")

df_trophies = spark.sql(f"""
    SELECT * FROM GamingPlatform_STG.dbo.trophies
    WHERE ModifiedDate > '{last_load_date}'
""")

df_user_activity = spark.sql(f"""
    SELECT * FROM GamingPlatform_STG.dbo.user_activity
    WHERE ModifiedDate > '{last_load_date}'
""")

df_user_basic = spark.sql(f"""
    SELECT * FROM GamingPlatform_STG.dbo.user_basic
    WHERE ModifiedDate > '{last_load_date}'
""")

df_user_contact = spark.sql(f"""
    SELECT * FROM GamingPlatform_STG.dbo.user_contact
    WHERE ModifiedDate > '{last_load_date}'
""")

df_user_location = spark.sql(f"""
    SELECT * FROM GamingPlatform_STG.dbo.user_location
    WHERE ModifiedDate > '{last_load_date}'
""")

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 6, Finished, Available, Finished, False)

## Create Views from all DataFrames

In [5]:
# Create Views from all DataFrames

df_cities.createOrReplaceTempView("vw_cities_stg")
df_countries.createOrReplaceTempView("vw_countries_stg")
df_game_genres.createOrReplaceTempView("vw_game_genres_stg")
df_game_metadata.createOrReplaceTempView("vw_game_metadata_stg")
df_game_prices.createOrReplaceTempView("vw_game_prices_stg")
df_game_sessions.createOrReplaceTempView("vw_game_sessions_stg")
df_game_titles.createOrReplaceTempView("vw_game_titles_stg")
df_orders.createOrReplaceTempView("vw_orders_stg")
df_states.createOrReplaceTempView("vw_states_stg")
df_trophies.createOrReplaceTempView("vw_trophies_stg")
df_user_activity.createOrReplaceTempView("vw_user_activity_stg")
df_user_basic.createOrReplaceTempView("vw_user_basic_stg")
df_user_contact.createOrReplaceTempView("vw_user_contact_stg")
df_user_location.createOrReplaceTempView("vw_user_location_stg")

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 7, Finished, Available, Finished, False)

## Create Dimensions

In [6]:
# create dim_user
df_dim_user = spark.sql("""
    SELECT 
        ub.user_id,
        ub.username,
        ub.status,
        uc.email,
        uc.phone,
        ua.created_at,
        ub.ModifiedDate
    FROM vw_user_basic_stg ub
    LEFT JOIN vw_user_contact_stg uc 
        ON ub.user_id = uc.user_id
    LEFT JOIN vw_user_activity_stg ua 
        ON ub.user_id = ua.user_id
""")

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 8, Finished, Available, Finished, False)

In [7]:
# create dim_game
df_dim_game = spark.sql("""
    SELECT 
        gt.game_id,
        gt.game_name,
        gg.genre as genre_name,
        gm.platform,
        gm.release_date,
        gm.rating,
        gp.price,
        gp.discount_percentage,
        gp.tax_percentage,
        gt.ModifiedDate
    FROM vw_game_titles_stg gt
    LEFT JOIN vw_game_genres_stg gg 
        ON gt.game_id = gg.game_id
    LEFT JOIN vw_game_metadata_stg gm 
        ON gt.game_id = gm.game_id
    LEFT JOIN vw_game_prices_stg gp 
        ON gt.game_id = gp.game_id
""")

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 9, Finished, Available, Finished, False)

In [8]:
# Creare dim_location
df_dim_location = spark.sql("""
    SELECT 
        c.city_id,
        c.city_name,
        s.state_name,
        co.country_name,
        co.region,
        c.ModifiedDate
    FROM vw_cities_stg c
    LEFT JOIN vw_states_stg s 
        ON c.state_id = s.state_id
    LEFT JOIN vw_countries_stg co 
        ON s.country_id = co.country_id
""")

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 10, Finished, Available, Finished, False)

In [9]:
# Creare dim_payment
df_dim_payment = spark.sql("""
    SELECT DISTINCT 
        payment_method,
        ModifiedDate
    FROM vw_orders_stg
""")

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 11, Finished, Available, Finished, False)

In [10]:
# Creare df_dim_device
df_dim_device = spark.sql("""
    SELECT DISTINCT 
       NULLIF(TRIM(device_type), 'N.A') AS device_type,
       ModifiedDate
    FROM vw_game_sessions_stg
""")

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 12, Finished, Available, Finished, False)

In [11]:
df_dim_trophy = spark.sql("""
    WITH DistinctTrophies AS (
        SELECT DISTINCT
            NULLIF(TRIM(trophy_name), 'N.A') AS trophy_name,
            NULLIF(TRIM(trophy_type), 'N.A') AS trophy_type,
            ModifiedDate
        FROM vw_trophies_stg
    ),
    NumberedTrophies AS (
        SELECT
            CAST(ROW_NUMBER() OVER (ORDER BY trophy_name, trophy_type) AS INT) AS trophy_id,
            trophy_name,
            trophy_type,
            ModifiedDate,
            -- Calculated Column
            CASE 
                WHEN trophy_type = 'Gold'   THEN 'Premium'
                WHEN trophy_type = 'Silver' THEN 'Advanced'
                ELSE 'Basic'
            END AS Trophy_Rank_Category
          
        FROM DistinctTrophies
    )
    SELECT *
    FROM NumberedTrophies
""")

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 13, Finished, Available, Finished, False)

## Create SK for Dimensions

In [12]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, max, col

# =========================================================
# USER DIMENSION
# =========================================================

max_user_sk = spark.table("dim_user") \
    .agg(max("user_sk")) \
    .collect()[0][0]

if max_user_sk is None:
    max_user_sk = 0

window_spec = Window.orderBy("user_id")

df_dim_user = df_dim_user.withColumn(
    "user_sk",
    row_number().over(window_spec) + max_user_sk+1
)

# =========================================================
# GAME DIMENSION
# =========================================================

max_game_sk = spark.table("dim_game") \
    .agg(max("game_sk")) \
    .collect()[0][0]

if max_game_sk is None:
    max_game_sk = 0

window_spec = Window.orderBy("game_id")

df_dim_game = df_dim_game.withColumn(
    "game_sk",
    row_number().over(window_spec) + max_game_sk+1
)

# =========================================================
# LOCATION DIMENSION
# =========================================================

max_location_sk = spark.table("dim_location") \
    .agg(max("location_sk")) \
    .collect()[0][0]

if max_location_sk is None:
    max_location_sk = 0

window_spec = Window.orderBy("city_id")

df_dim_location = df_dim_location.withColumn(
    "location_sk",
    row_number().over(window_spec) + max_location_sk+1
)

# =========================================================
# PAYMENT DIMENSION
# =========================================================

max_payment_sk = spark.table("dim_payment") \
    .agg(max("payment_sk")) \
    .collect()[0][0]

if max_payment_sk is None:
    max_payment_sk = 0

window_spec = Window.orderBy("payment_method")

df_dim_payment = df_dim_payment.withColumn(
    "payment_sk",
    row_number().over(window_spec) + max_payment_sk+1
)

# =========================================================
# DEVICE DIMENSION
# =========================================================

max_device_sk = spark.table("dim_device") \
    .agg(max("device_sk")) \
    .collect()[0][0]

if max_device_sk is None:
    max_device_sk = 0

window_spec = Window.orderBy("device_type")

df_dim_device = df_dim_device.withColumn(
    "device_sk",
    row_number().over(window_spec) + max_device_sk+1
)

# =========================================================
# TROPHY DIMENSION
# =========================================================

max_trophy_sk = spark.table("dim_trophy") \
    .agg(max("trophy_sk")) \
    .collect()[0][0]

if max_trophy_sk is None:
    max_trophy_sk = 0

window_spec = Window.orderBy("trophy_id")

df_dim_trophy = df_dim_trophy.withColumn(
    "trophy_sk",
    row_number().over(window_spec) + max_trophy_sk+1
)

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 14, Finished, Available, Finished, False)

## Applay Incremental-load

In [13]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
df_dim_user.cache()

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 15, Finished, Available, Finished, False)

DataFrame[user_id: int, username: string, status: string, email: string, phone: string, created_at: timestamp, ModifiedDate: timestamp, user_sk: int]

In [14]:
DeltaTable.forName(spark, "Gaming_Platform_DW.dbo.dim_device") \
    .alias("t") \
    .merge(
        df_dim_device.alias("s"),
        "t.device_type = s.device_type"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 16, Finished, Available, Finished, False)

In [15]:
DeltaTable.forName(spark, "Gaming_Platform_DW.dbo.dim_game") \
    .alias("t") \
    .merge(
        df_dim_game.alias("s"),
        "t.game_id = s.game_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 17, Finished, Available, Finished, False)

In [16]:
DeltaTable.forName(spark, "Gaming_Platform_DW.dbo.dim_location") \
    .alias("t") \
    .merge(
        df_dim_location.alias("s"),
        "t.city_id = s.city_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 18, Finished, Available, Finished, False)

In [17]:
DeltaTable.forName(spark, "Gaming_Platform_DW.dbo.dim_payment") \
    .alias("t") \
    .merge(
        df_dim_payment.alias("s"),
        "t.payment_method = s.payment_method"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 19, Finished, Available, Finished, False)

In [18]:
DeltaTable.forName(spark, "Gaming_Platform_DW.dbo.dim_trophy") \
    .alias("t") \
    .merge(
        df_dim_trophy.alias("s"),
        """t.trophy_name = s.trophy_name
        AND t.trophy_type = s.trophy_type"""
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 20, Finished, Available, Finished, False)

In [19]:


DeltaTable.forName(spark, "Gaming_Platform_DW.dbo.dim_user") \
    .alias("t") \
    .merge(
        df_dim_user.alias("s"),
        "t.user_id = s.user_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 21, Finished, Available, Finished, False)

# **Create Facts**

## Fact Orders

In [20]:
# Load all Dimension tables from Gaming_Platform_DW

df_dim_date     = spark.sql("SELECT * FROM Gaming_Platform_DW.dbo.dim_date")
df_dim_device   = spark.sql(f"SELECT * FROM Gaming_Platform_DW.dbo.dim_device WHERE ModifiedDate > '{last_load_date}'")
df_dim_game     = spark.sql(f"SELECT * FROM Gaming_Platform_DW.dbo.dim_game WHERE ModifiedDate > '{last_load_date}'")
df_dim_location = spark.sql(f"SELECT * FROM Gaming_Platform_DW.dbo.dim_location WHERE ModifiedDate > '{last_load_date}'")
df_dim_payment  = spark.sql(f"SELECT * FROM Gaming_Platform_DW.dbo.dim_payment WHERE ModifiedDate > '{last_load_date}'")
df_dim_trophy   = spark.sql(f"SELECT * FROM Gaming_Platform_DW.dbo.dim_trophy WHERE ModifiedDate > '{last_load_date}'")
df_dim_user     = spark.sql(f"SELECT * FROM Gaming_Platform_DW.dbo.dim_user WHERE ModifiedDate > '{last_load_date}'")

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 22, Finished, Available, Finished, False)

In [21]:
# Create Views from all Dimension DataFrames

df_dim_date.createOrReplaceTempView("vw_dim_date")
df_dim_device.createOrReplaceTempView("vw_dim_device")
df_dim_game.createOrReplaceTempView("vw_dim_game")
df_dim_location.createOrReplaceTempView("vw_dim_location")
df_dim_payment.createOrReplaceTempView("vw_dim_payment")
df_dim_trophy.createOrReplaceTempView("vw_dim_trophy")
df_dim_user.createOrReplaceTempView("vw_dim_user")

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 23, Finished, Available, Finished, False)

In [22]:
df_fact_orders = spark.sql(f"""
    SELECT 
        COALESCE(du.user_sk, -1)        AS user_sk,
        COALESCE(dg.game_sk, -1)        AS game_sk,
        CAST(o.order_date AS TIMESTAMP) AS full_date,
        COALESCE(dl.location_sk, -1)    AS location_sk,
        COALESCE(dp.payment_sk, -1)     AS payment_sk,
        o.order_id,
        o.price,
        o.quantity,
        o.discount_amount,
        o.tax_amount,
        o.total_amount,
        o.is_refunded,
        o.refund_date,
        o.order_date,
        o.ModifiedDate,
        -- Calculated Columns
        CASE WHEN o.total_amount > 90 THEN 'High' ELSE 'Low' END          AS Revenue_Category,
        CASE WHEN o.discount_amount > 0 THEN 'Discounted' ELSE 'No Discount' END AS Discount_Status,
        CASE WHEN o.is_refunded = TRUE THEN 'Refunded' ELSE 'Not Refunded' END   AS Refund_Status

    FROM vw_orders_stg o
    LEFT JOIN vw_dim_user du 
        ON o.user_id = du.user_id
    LEFT JOIN vw_dim_game dg 
        ON o.game_id = dg.game_id
    LEFT JOIN vw_user_location_stg ul 
        ON o.user_id = ul.user_id
    LEFT JOIN vw_dim_location dl 
        ON ul.city_id = dl.city_id
    LEFT JOIN vw_dim_payment dp 
        ON o.payment_method = dp.payment_method
""")

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 24, Finished, Available, Finished, False)

In [23]:
max_fact_orders = spark.table("Gaming_Platform_DW.dbo.fact_orders") \
    .agg(max("order_sk")) \
    .collect()[0][0]

if max_fact_orders is None:
    max_fact_orders = 0

window_spec = Window.orderBy("order_id")

df_fact_orders= df_fact_orders.withColumn(
    "order_sk",
    row_number().over(window_spec) + max_fact_orders
)


StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 25, Finished, Available, Finished, False)

In [24]:
DeltaTable.forName(spark, "Gaming_Platform_DW.dbo.fact_orders") \
    .alias("t") \
    .merge(
        df_fact_orders.alias("s"),
        "t.order_id = s.order_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 26, Finished, Available, Finished, False)

## Fact_Gam_stats

In [25]:
df_fact_game_stats = spark.sql("""
    SELECT 
        COALESCE(du.user_sk, -1)          AS user_sk,
        COALESCE(dg.game_sk, -1)          AS game_sk,
        CAST(gs.session_date AS TIMESTAMP) AS full_date,
        COALESCE(dl.location_sk, -1)      AS location_sk,
        COALESCE(dd.device_sk, -1)        AS device_sk,
        gs.session_id,
        gs.hours_played,
        gs.ModifiedDate,
        -- Calculated Column
        CASE 
            WHEN gs.hours_played > 16 THEN 'Hardcore'
            WHEN gs.hours_played > 10 THEN 'Regular'
            ELSE 'Casual'
        END AS Player_Activity_Level

    FROM vw_game_sessions_stg gs
    LEFT JOIN vw_dim_user du 
        ON gs.user_id = du.user_id
    LEFT JOIN vw_dim_game dg 
        ON gs.game_id = dg.game_id
    LEFT JOIN vw_user_location_stg ul 
        ON gs.user_id = ul.user_id
    LEFT JOIN vw_dim_location dl 
        ON ul.city_id = dl.city_id
    LEFT JOIN vw_dim_device dd 
        ON gs.device_type = dd.device_type
""")

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 27, Finished, Available, Finished, False)

In [26]:
max_fact_game_stats = spark.table("Gaming_Platform_DW.dbo.fact_game_stats") \
    .agg(max("session_sk")) \
    .collect()[0][0]

if max_fact_game_stats is None:
    max_fact_game_stats = 0

window_spec = Window.orderBy("session_id")

df_fact_game_stats = df_fact_game_stats.withColumn(
    "session_sk",
    row_number().over(window_spec) + max_fact_game_stats
)

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 28, Finished, Available, Finished, False)

In [27]:
DeltaTable.forName(spark, "Gaming_Platform_DW.dbo.fact_game_stats") \
    .alias("t") \
    .merge(
        df_fact_game_stats.alias("s"),
        "t.session_id = s.session_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 29, Finished, Available, Finished, False)

## fact_trophies

In [28]:
df_fact_trophies = spark.sql("""
    SELECT 
        COALESCE(du.user_sk, -1)           AS user_sk,
        COALESCE(dg.game_sk, -1)           AS game_sk,
        COALESCE(dt.trophy_sk, -1)         AS trophy_sk,
        CAST(t.earned_date AS TIMESTAMP)   AS full_date,
        COALESCE(dl.location_sk, -1)       AS location_sk,
        t.trophy_id,
        t.earned_date,
        t.ModifiedDate

    FROM vw_trophies_stg t
    LEFT JOIN vw_dim_user du 
        ON t.user_id = du.user_id
    LEFT JOIN vw_dim_game dg 
        ON t.game_id = dg.game_id
    LEFT JOIN vw_dim_trophy dt 
        ON t.trophy_name = dt.trophy_name
        AND t.trophy_type = dt.trophy_type
    LEFT JOIN vw_user_location_stg ul 
        ON t.user_id = ul.user_id
    LEFT JOIN vw_dim_location dl 
        ON ul.city_id = dl.city_id
""")

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 30, Finished, Available, Finished, False)

In [29]:
max_fact_trophies = spark.table("Gaming_Platform_DW.dbo.fact_trophies") \
    .agg(max("trophy_event_sk")) \
    .collect()[0][0]

if max_fact_trophies is None:
    max_fact_trophies = 0

window_spec = Window.orderBy("trophy_id")

df_fact_trophies = df_fact_trophies.withColumn(
    "trophy_event_sk",
    row_number().over(window_spec) + max_fact_trophies
)

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 31, Finished, Available, Finished, False)

In [30]:
DeltaTable.forName(spark, "Gaming_Platform_DW.dbo.fact_trophies") \
    .alias("t") \
    .merge(
        df_fact_trophies.alias("s"),
        "t.trophy_id = s.trophy_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 32, Finished, Available, Finished, False)

## Stop Spark Session

In [31]:
# mssparkutils.session.stop()

StatementMeta(, ee2d5a59-eff0-45e3-a21f-e4a70ef2848d, 33, Finished, Available, Finished, False)